[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frankausberlin/deen-lupysta/blob/main/scripts/dash.ipynb)

In [14]:
#@markdown <font size='+2' color='#005F6A'>**OpenRouterModels**</font><br>
#@markdown * Uses **openrouter-cli**-pip to get model infos from OpenRouter. It **must be installed**: `uv tool install openrouter-cli` or `pip install openrouter-cli`
#@markdown * You must insert your **OpenRouter API-key** first with: `openrouter-cli configure`'.
#@markdown * You can **specify** a comma-separated **list of frontier models** to be **scanned**, whose **latest versions** will be displayed in the **overview**.
from ipywidgets import Button, Box, Layout, Textarea, VBox, HTML, HBox
import subprocess, json, pprint

class ORMButtonBox ():
  def __init__ (self, descriptions, clicker=None, maxchar=60, color=None):
    # remember stuff - list to set
    self.descriptions = descriptions if len (descriptions) == len (set (descriptions)) else set (descriptions)
    self.clicker, self.maxchar, self.color, self.position, self.button = clicker, maxchar, color, -1, None
    self._original_colors = {} # Dictionary to store original colors

    # make buttons
    self.buttons = [Button (description = i if len (i) <= maxchar else f'{i[:(maxchar-3)]}...',
                            layout      = Layout(width='auto', height='22px'),
                            tooltip     = f'{i}')
                    for i in descriptions]

    # put them in a box / bind event
    self.widget = Box (layout   = Layout (display='flex', flex_flow='wrap'),
                       children = self.buttons)
    for button in self.buttons:
      self._original_colors[button] = button.style.button_color # Store original color
      button.on_click (self._clicker)

  # An alternative for the Tab widget
  def _clicker (self, b):
    # search clicked button and remember
    self.position, self.button = [(i,but) for i, but in enumerate(self.buttons) if b == but][0]

    # unselect (color) all buttons and select new (list of colors here: https://www.quackit.com/css/css_color_codes.cfm)
    self.unselect()
    if self.color: self.buttons[self.position].style={'button_color':self.color}
    self.buttons[self.position].layout.border = '1px solid black'

    # fire event
    if self.clicker: self.clicker (self)

  def unselect (self):
    # unselect all buttons and restore original color
    for b in self.buttons:
      b.layout.border = ''
      b.style = {'button_color': self._original_colors.get(b, None)} # Restore original color, default to None if not found

def selectOrganization (bbox):
  global selectedOrga
  selectedOrga =  bbox.button.tooltip
  org_models = [m for m in json_list if m['id'].split('/')[0] == selectedOrga]
  modelsBox = ORMButtonBox ([i['id'].replace (bbox.button.tooltip+'/','') for i in org_models], clicker=selectModel, color='powderblue')
  widgetList.children=(widgetList.children[0], widgetList.children[1], modelsBox.widget)

def selectModel (bbox):
  for model in json_list:
    if model['id'] == selectedOrga+'/'+bbox.button.tooltip:
      if len (widgetList.children) < 4:
        widgetList.children = (*widgetList.children, Textarea (layout=Layout(width='auto', height='500px')))
      widgetList.children[3].value = pprint.pformat(model, width=160)

# ___________________________________________________________
#|______________________hello_component______________________|
try:
  json_list = json.loads(subprocess.run(['openrouter-cli','models', '--raw'], capture_output=True, text=True, check=True).stdout)
  tab_list = subprocess.run(['openrouter-cli','models'], capture_output=True, text=True, check=True).stdout.split('\n')
except: json_list = []

if json_list:

  # generate frontier models overview
  frontier_models = "gpt-5, claude-opus-4, claude-sonnet-4, gemini-3, deepseek-v4, minimax-m2, kimi-k2, glm-5, qwen3, mimo-v2, grok-4, mistral-medium-3-5"  #@param {type:"string"}
  ignore_frontier_models_with = "8b, 27b, 35b, a3b, -lite, -image, -micro, -nano, qwen3.6-flash"  #@param {type:"string"}
  ignore = ignore_frontier_models_with.replace(' ', '').split(',')

  # build the list with tuples ('versionnr', 'model-id') for the frontier models
  frontiers = frontier_models.replace(' ', '').split(',')

  # correct filter
  bad_id = lambda id: sum([f in id for f in ignore])

  frontier_version_id_list = [(j['id'].split('/')[-1].split(f[:-1])[-1].split('-')[0] , j['id'])
                              for f in frontiers for j in json_list
                                if f in j['id'] and
                                  ':free' not in j['id'] and
                                  j['id'].split('/')[-1].split(f[:-1])[-1].split('-')[0][-1] in '0123456789' and
                                  not bad_id (j['id'])]

  # build the frontier model overview
  tmp = []
  for frontier in frontiers:
    match = sorted ([f for f in frontier_version_id_list if frontier in f[1]])
    if not match: continue # catch empty lists due to rigorous filtering

    show_version = match[-1][0]
    versions = [f for f in match if show_version == f[0]]
    hm = f"<font color=blue size=3><b>{versions[0][1].split('/')[0]}</b></font><hr>"
    for version in versions:
      hm += f"<b>{version[1].split('/')[1]}</b><br>"
      idj = [j for j in json_list if j['id'] == version[1]][0]
      r, w  = f"{(float (idj['pricing']['prompt'])*1000000):.2f}", f"{(float (idj['pricing']['completion'])*1000000):.2f}"
      cr = f"{(float(idj['pricing']['input_cache_read'])*1000000):.2f}" if 'input_cache_read' in idj['pricing'] else ' '
      cw = f"{(float(idj['pricing']['input_cache_write'])*1000000):.2f}" if 'input_cache_write' in idj['pricing'] else ' '
      hm += f"Context: {idj['context_length']}<br>Price: {r} / {w}<br>[Cache: {cr} / {cw}]<br>"

    tmp.append (HTML(value=hm[:-4], layout={"border":"2px solid yellow"}))
  overview_panel = HBox (children=tmp)

  # build orga / model selector
  selectedOrga      = None
  organization_list = sorted (set ([j['id'].split('/')[0] for j in json_list if '/' in j['id']]))
  organizationsBox  = ORMButtonBox (organization_list, clicker=selectOrganization, color='powderblue')
  title             = HTML (value="<font color=green size=4><b>Overview Frontier Models Latest")
  widgetList        = VBox (children=[organizationsBox.widget,HTML(value='<hr>')])

  display (VBox (children=[title,overview_panel]), widgetList)

In [ ]:
#@markdown <font size='+2' color='#005F6A'>**Hosting: CudaOnHostTest**</font><br>Check if cuda runing on host and is available for **pytorch, tensorflow, jax** and **numba**.

import os, warnings

# if you are using > Python 3.11:
# with warnings.catch_warnings(action="ignore"):

with warnings.catch_warnings():
  warnings.simplefilter("ignore")

  try:
    import torch
    cuda_pytorch = torch.cuda.is_available()
  except: cuda_pytorch = False

  try:
    import tensorflow as tf
    cuda_tensorflow = ':GPU:' in str(tf.config.list_physical_devices('GPU'))
  except: cuda_tensorflow = False

  try:
    import jax
    cuda_jax = len(jax.devices()) > 0
  except: cuda_jax = False

  try:
    from numba import cuda
    cuda_numba = cuda.is_available()
  except: cuda_numba = False

  try:
    from IPython.display import clear_output
    clear_output ()
  except: pass


  hasNvidiaSmi  = os.system ('nvidia-smi --version >/dev/null 2>&1') == 0
  if hasNvidiaSmi:
    print (f"pytorch-cuda:     \x1b[92mTrue\x1b[0m") if cuda_pytorch     else print (f"pytorch-cuda:     \x1b[91mFalse\x1b[0m")
    print (f"tensorflow-cuda:  \x1b[92mTrue\x1b[0m") if cuda_tensorflow  else print (f"tensorflow-cuda:  \x1b[91mFalse\x1b[0m")
    print (f"jax-cuda:         \x1b[92mTrue\x1b[0m") if cuda_jax         else print (f"jax-cuda:         \x1b[91mFalse\x1b[0m")
    print (f"numba-cuda:       \x1b[92mTrue\x1b[0m\n") if cuda_numba       else print (f"numba-cuda:       \x1b[91mFalse\x1b[0m")
    !nvidia-smi

  else:
    print ('\x1b[91mno cuda on host\x1b[0m')


In [ ]:
#@markdown <font size='+2' color='#005F6A'>**Hosting: Ollama on vast.ai/colab**<br></font>
#@markdown * This snippet is inspired by [this documentation](https://docs.vast.ai/google-colab) and this [article](https://medium.com/@abonia/running-ollama-in-google-colab-free-tier-545609258453). Using [colab-xterm](https://github.com/InfuseAI/colab-xterm), [Ollama](https://ollama.com/download) and [Ollama for python](https://github.com/ollama/ollama-python) - models can be found [here](https://ollama.com/search).
#@markdown * You have three behaviors:<br>1. <font color=green><b>Local runtime</b></font> (port 8888): You will receive instructions, and the SSH key will be copied to the clipboard.<br>2. <font color=green><b>Vast.ai runtime</b></font> (port 8080): This is where Ollama is installed and started, and you will receive instructions.<br>3. <font color=green><b>Colab runtime</b></font>: Installs Ollama on Colab (you should use a GPU runtime) - no ngrok and vast.ai needed.
#@markdown * Just run the cell, the **correct behavior will be executed automatically**.
import sys, os
from IPython.display import clear_output
from ipywidgets import Button, Textarea, HBox, Text, Layout
from typing_extensions import final

test_prompt="""Here is a classic math text problem: Uwe and Peter are talking.
Uwe says: "If you give me an apple from yours, then we will both have the same number of apples."
Then Peter says, "But if you give me one of yours, I'll have twice as many as you."
How many apples does each of them have?"""

script = """
   jupyter serverextension enable --py jupyter_http_over_ws
   jupyter notebook \\
     --NotebookApp.allow_origin='https://colab.research.google.com' \\
     --port=8080 \\
     --NotebookApp.port_retries=0 \\
     --allow-root \\
     --NotebookApp.allow_remote_access=True \\
     --NotebookApp.allow_credentials=True &
"""

# Analyse situation
isColab   = (os.getenv("COLAB_RELEASE_TAG") != None)
isRoot    = (os.path.expanduser('~') == '/root')
hasXclip  = (os.system ('xclip -V >/dev/null 2>&1') == 256)

# on local jupyter
if not isRoot:
  # check key exists
  if not os.path.exists (os.path.expanduser('~')+'/.ssh/id_rsa.pub'):
    print  ('\x1b[1;91mplease generate id_rsa.pub first\x1b[0m (https://ngrok.com/docs/getting-started/)')
  else:
    # show key in textarea
    ta = Textarea  (layout={'width':'1000px', 'height':'100px'})
    key = !cat ~/.ssh/id_rsa.pub
    ta.value = key[0]
    display (ta)

    # copy to clipboard
    if hasXclip and os.system ('cat ~/.ssh/id_rsa.pub | xclip -selection clipboard >/dev/null 2>&1') == 0:
      print ('\x1b[1;92m✓ copied ssh key to clipboard\x1b[0m')
    else:
      print ('\x1b[1;91mplease copy the key manualy\x1b[0m')

    # output instructions
    print ("\n\x1b[1;94m1. start vast.ai with the colab-template:\x1b[0m\n   https://cloud.vast.ai/?template_id=8d383ad48fff4012d42806e4781020ef")
    print ("\n\x1b[1;94m2. on vast.ai under 'instances' press the '>_connect' button\x1b[0m")
    print ("\n\x1b[1;94m3. on the dialog add your ssh key (from clipboard)\x1b[0m")
    print ("\n\x1b[1;94m4. copy the ssh connect command\x1b[0m")
    print ("\n\x1b[1;94m5. open a terminal on your machine and past command and confirm the connection 'yes'\x1b[0m")
    print (f"\n\x1b[1;94m6. start jupyter on vast.ai:\x1b[0m\n{script}")
    print ("\n\x1b[1;94m7. copy the url with the token, select 'connect with local runtime' in colab and past the url\x1b[0m\n\n   Be aware that you have two local runtimes:\n      - the runtime on port 8888 on your machine\n      - the runtime on port 8080 on the vast.ai machine (ngrok)")
    print ("\n\x1b[1;94m8. run this cell again on vast.ai runtime to install ollama and get instructions")

# on vast.ai-jupyter .ssh
if isRoot and (not isColab):
  # install ollama
  !command -v ollama >/dev/null 2>&1 || curl -fsSL https://ollama.com/install.sh | sh
  try:     from ollama import chat
  except:  os.system ('pip install -q ollama')
  finally: from ollama import chat

  # instructions
  clear_output()
  print ("\n\x1b[1;94m1. Run this in a cell to start Ollama: !ollama serve")
  print ("\n\x1b[1;94m2. You can use it in the terminal with the ssh connection")
  print ("\n\x1b[1;94m2. run/pull a model with:\x1b[0m\n   eg: ollama run/pull gemma3:12b")
  print ("\n\x1b[1;94m3. you can select a model here:\x1b[0m\n   https://ollama.com/search ")
  print (f"\n\x1b[1;94m4. here is a little prompt to test:\x1b[0m\n{test_prompt}")
  !ollama serve

# on colab
if isColab:

  print ('\x1b[4;10mIf you want to use the \x1b[1;92mvast.ai runtime\x1b[0m\x1b[4;10m do this:\x1b[0m\n')
  print ("   \x1b[1;94m1. start vast.ai with the colab-template:\x1b[0m\n   https://cloud.vast.ai/?template_id=8d383ad48fff4012d42806e4781020ef")
  print ("   \x1b[1;94m2. on vast.ai under 'instances' press the '>_connect' button\x1b[0m")
  print ("   \x1b[1;94m3. on the dialog add your ssh key (`cat ~/.ssh/id_rsa.pub`)\x1b[0m")
  print ("   \x1b[1;94m4. copy the ssh connect command\x1b[0m")
  print ("   \x1b[1;94m5. open a terminal on your machine and past command and confirm the connection 'yes'\x1b[0m")
  print (f"   \x1b[1;94m6. start jupyter on vast.ai:\x1b[0m\n{script}")
  print (f"   \x1b[1;94m7. disconnect and delete the colab runtime")
  print ("   \x1b[1;94m8. copy the url with the token, select 'connect with local runtime' in colab and past the url\x1b[0m")
  print ("   \x1b[1;94m9. run this cell again on vast.ai runtime to install ollama and get instructions\x1b[0m")
  print (f"\n\x1b[1;94mhere is a little prompt to test:\x1b[0m\n{test_prompt}")
  print ('\n\n\x1b[4;10mIf you want to use the \x1b[1;92mcolab runtime\x1b[0m\x1b[4;10m (only 16gb vram) do this:\x1b[0m\n')
  print ('   \x1b[91mTo copy commands: ctrl-c in the output area (here) and ctrl-shift-v in xterm!!!')
  print ('   Sometimes xterm has absurdly long response times (~30 sec.) - be patient\x1b[0m')
  print ('   1. \x1b[1;94mStart xterm\x1b[0m:  click the button')

  print ('   2. \x1b[1;94mInstall ollama\x1b[0m:  curl https://ollama.ai/install.sh | sh')
  print ('   3. \x1b[1;94mStart ollama\x1b[0m:    ollama serve & # press enter after done to get prompt back')
  print ('   4. \x1b[1;94mLoad model\x1b[0m:      ollama pull deepseek-r1 # deepseek-v2 deepseek-r1:14b qwen2.5-coder:14b olmo2:13b')
  print ('   5. \x1b[1;94mRun model\x1b[0m:       ollama run deepseek-r1')


